# Determinism under parallelism

The contract is byte-identical: the same table and the same
configuration produce the same report bytes, on any thread count, on any
run. Determinism is not a nicety here — an agent that profiles a table
twice and gets two different answers stops trusting both — and it is
achieved structurally, not by trying hard.

## The rules

1. **No wall clock in results.** Timing exists only as the opt-in
   `deadline_seconds`, and when it fires, what was skipped is recorded —
   the deadline changes *which passes ran*, never *what a pass found*.
2. **Seeds everywhere randomness is used.** The ξ tie-breaking shuffle,
   the GBM holdout shuffles, the stratified sampler's stratum offsets —
   all are fixed arithmetic or fixed-seed generators derived from
   table shape and column indices, never from entropy sources.
3. **Sorted, keyed collections in the report path.** The report tree is
   built from `BTreeMap`s and sorted vectors; the JSON writer walks them
   in key order. Key-order iteration is insertion-order-independent.
4. **Stable tie-breaking everywhere.** Every sort in the pipeline that
   can see ties has a final tie-break on a stable key (a column name, an
   index) so "equal" candidates keep a fixed order.
5. **Deterministic reduction of parallel results.** Rayon work happens
   per column, per pair, per target — results are collected into
   position-indexed containers or merged through ordered maps, never
   accumulated into a shared float in arrival order. Where floating-point
   sums matter (concentration), the summation order is the sorted key
   order, not the parallel traversal order.

## Why parallelism does not break it

The pattern that makes rayon safe here: *partition, compute, collect by
position*. A parallel pass computes one result per work item (one pair,
one target, one candidate set), rayon returns them in input order, and
any aggregation happens afterwards over the ordered collection. No pass
reduces into an accumulator whose value depends on scheduling. Where a
global ranking happens (dependence findings, rule selection), it sorts
by (score, stable keys) — the sort that deduplicates ties — before any
truncation, so which items survive a cap is schedule-independent.

The stratified sampler deserves its own note: `sample_rows(n, cap)`
picks row `start + mix(stratum) mod stratum_width` per stratum. A plain
stride (`row = k·j`) would also be deterministic — and would resonantly
sample one weekday out of a weekly cycle. The hashed offset is exactly
as deterministic and does not alias ([](../methodology/sampling.ipynb)).

## What opt-in nondeterminism looks like

The deadline is the single opt-in. It is checked between passes; the
pass that observes it completes, later passes are skipped, and coverage
records `"deadline reached"` with the pass names. A report produced
under a deadline still contains only fully computed findings — but it is
a different report from the untimed one, and it should travel with its
coverage block when it travels. Everything else — thread counts, machine
speed, chunk layouts of the input — leaves the bytes untouched.

In [ ]:
import polars as pl

import onceover

df = pl.DataFrame({
    "a": [1, 2, 2, 3, None, 3],
    "b": [1.5, 2.5, None, 2.5, 7.0, 1.5],
})
first = onceover.profile(df)
second = onceover.profile(df, budget="standard")
print("byte-identical reruns:", first.to_json() == second.to_json())